In [ ]:
pip install mne

In [ ]:
import mne

mne.utils.set_config('MNE_USE_CUDA', 'true')

In [ ]:
# --- IMPORTACIONES ---
import numpy as np
import mne
import scipy.io
import scipy.interpolate
import plotly.express as px
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from google.colab import drive
import os

# --- 1. CONFIGURACIÓN DE GOOGLE COLAB Y RUTAS ---
drive.mount('/content/drive')

# Ruta base (donde están tus archivos .mat)
RUTA_BASE = "/content/drive/MyDrive/datos ECoG"

# Nueva carpeta para guardar resultados
RUTA_GUARDADO = os.path.join(RUTA_BASE, "datos_procesados")

# Crear carpeta si no existe
os.makedirs(RUTA_GUARDADO, exist_ok=True)
print("Carpeta lista en:", RUTA_GUARDADO)



# --- 2. HIPERPARÁMETROS Y CONSTANTES ---
FREQ_INF, FREQ_SUP = 40, 300     # Límites inferior y superior de filtrado
NUM_WAVELETS = 40                # Número de wavelets en el rango de frecuencia
FS_SUBMUESTREO = 100             # Frecuencia de muestreo deseada (100 Hz)
retraso_temporal_segs = 0.2      # Hiperparámetro de retraso temporal
fs_actual = FS_SUBMUESTREO

def redimensionar_columna_datos_ecog(senal_multicanal: np.ndarray):

    # Transpone la señal: de (tiempo, características) a (características, tiempo)

    return senal_multicanal.T

def filtrar_datos_ecog(senal_multicanal: np.ndarray, fs=1000, freq_red_electrica=50):
    """
    Eliminación de armónicos y filtrado de frecuencias.
    :param senal_multicanal: Señal multicanal inicial.
    :param fs: Frecuencia de muestreo original.
    :param freq_red_electrica: Frecuencia de la red eléctrica (para eliminar ruido).
    :return: Señal filtrada.
    """
    armonicos = np.array([i * freq_red_electrica for i in range(1, (fs // 2) // freq_red_electrica)])

    print("Iniciando filtrado...")
    senal_filtrada = mne.filter.filter_data(senal_multicanal, fs, l_freq=FREQ_INF, h_freq=FREQ_SUP)
    print("Frecuencias de ruido eliminadas...")

    senal_sin_ruido_electrico = mne.filter.notch_filter(senal_filtrada, fs, freqs=armonicos)
    print("Ruido de la línea eléctrica eliminado...")

    return senal_sin_ruido_electrico

def normalizar(senal_multicanal: np.ndarray, retornar_valores = None):
    """
    Estandarización y eliminación de la mediana de cada canal.
    """
    print("Normalizando...")
    medias = np.mean(senal_multicanal, axis=1, keepdims=True)
    desviaciones_std = np.std(senal_multicanal, axis=1, keepdims=True)

    datos_transformados = (senal_multicanal - medias) / desviaciones_std
    promedio_comun = np.median(datos_transformados, axis=0, keepdims=True)

    datos_transformados = datos_transformados - promedio_comun

    if retornar_valores:
        return datos_transformados, (medias, desviaciones_std)
    print("Normalizado...")
    return datos_transformados

def calcular_espectrogramas(senal_multicanal: np.ndarray, fs=1000,
                          frecuencias=np.logspace(np.log10(FREQ_INF), np.log10(FREQ_SUP), NUM_WAVELETS),
                          tipo_salida='power'):
    """
    Calcula los espectrogramas usando transformadas wavelet.
    """
    num_canales = len(senal_multicanal)

    print("Calculando wavelets...")
    espectrogramas = mne.time_frequency.tfr_array_morlet(
        senal_multicanal.reshape(1, num_canales, -1),
        sfreq=fs, freqs=frecuencias, output=tipo_salida, verbose=10, n_jobs=6
    )[0]

    print("Espectrograma wavelet calculado...")
    return espectrogramas

def reducir_muestreo_espectrogramas(espectrogramas: np.ndarray, fs_actual=1000, hz_necesarios=FREQ_SUP, nueva_fs=None):
    """
    Reduce la frecuencia de muestreo de los espectrogramas (downsampling).
    """
    print("Reduciendo la frecuencia del espectrograma...")
    if nueva_fs is None:
        nueva_fs = hz_necesarios * 2

    coeficiente_reduccion = fs_actual // nueva_fs
    assert coeficiente_reduccion > 1

    espectrograma_reducido = espectrogramas[:, :, ::coeficiente_reduccion]
    print("Espectrograma reducido...")
    return espectrograma_reducido

def normalizar_espectrogramas_a_db(espectrogramas: np.ndarray, convertir=False):
    """
    Conversión opcional a decibelios (db).
    """
    if convertir:
        return np.log10(espectrogramas + 1e-12)
    else:
        return espectrogramas

def interpolar_flexion_dedos(flexion_dedos, fs_actual=1000, fs_real=25, hz_necesarios=FS_SUBMUESTREO, tipo_interpolacion='cubic'):
    """
    Interpolación de la grabación del movimiento de los dedos para coincidir con la nueva frecuencia de muestreo.
    """
    print("Interpolando flexión de dedos...")
    proporcion_reduccion = fs_actual // fs_real
    print("Calculando valores de frecuencia real...")

    flexion_dedos_fs_real = flexion_dedos[:, ::proporcion_reduccion]
    flexion_dedos_fs_real = np.c_[flexion_dedos_fs_real, flexion_dedos_fs_real.T[-1]]

    proporcion_aumento = hz_necesarios // fs_real
    tiempos = np.asarray(range(flexion_dedos_fs_real.shape[1])) * proporcion_aumento

    print("Creando funciones de interpolación...")
    funciones_flexion_dedos_interpolada = [scipy.interpolate.interp1d(tiempos, canal_dedo, kind=tipo_interpolacion)
                                      for canal_dedo in flexion_dedos_fs_real]

    tiempos_hz_necesarios = np.asarray(range(flexion_dedos_fs_real.shape[1] * proporcion_aumento)[:-proporcion_aumento])

    print("Interpolando con la frecuencia requerida...")
    flexion_dedos_interpolada = np.array([[func(t) for t in tiempos_hz_necesarios]
                                         for func in funciones_flexion_dedos_interpolada])
    return flexion_dedos_interpolada

def recortar_por_retraso_temporal(flexion_dedos: np.ndarray, espectrogramas: np.ndarray, retraso_temporal_seg: float, fs: int):
    """
    Toma en cuenta el retraso entre las ondas cerebrales y los movimientos físicos.
    """
    retraso_temporal = int(retraso_temporal_seg * fs)

    flexion_dedos_recortada = flexion_dedos[..., retraso_temporal:]
    espectrogramas_recortados = espectrogramas[..., :espectrogramas.shape[2] - retraso_temporal]

    return flexion_dedos_recortada, espectrogramas_recortados

def visualizar_senal(senal_multicanal: np.ndarray, num_canal: int, num_segundo: int, fs=FS_SUBMUESTREO):
    """
    Función para visualizar una sección de la señal multicanal.
    """
    df_canal = pd.DataFrame(data=np.asarray([np.asarray(range(fs)),
                                             senal_multicanal[num_canal][num_segundo*fs:num_segundo*fs+fs]]).T,
                            index=range(fs), columns=["t", "V"])

    figura = px.line(df_canal, x="t", y="V", title=f'Canal_{num_canal}')
    figura.show()



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Carpeta lista en: /content/drive/MyDrive/datos ECoG/datos_procesados


In [ ]:
# --- 4. BUCLE PRINCIPAL DE PROCESAMIENTO ---

# Definimos los sujetos y sus respectivos canales según tus indicaciones
sujetos_info = [
    {"id": 1, "canales": 62},
    {"id": 2, "canales": 48},
    {"id": 3, "canales": 64}
]

for sujeto in sujetos_info:
    sub_id = sujeto["id"]
    num_canales = sujeto["canales"]

    print(f"SUJETO {sub_id} ({num_canales} canales) ")


    ruta_comp = f'{RUTA_BASE}/sub{sub_id}_comp.mat'
    ruta_test = f'{RUTA_BASE}/sub{sub_id}_testlabels.mat'

    if not os.path.exists(ruta_comp):
        print(f"Archivo {ruta_comp} no encontrado. Saltando este sujeto.")
        continue

# --- A. DATOS DE ENTRENAMIENTO ---
    print("\n>> Procesando Entrenamiento (Train)...")
    datos = scipy.io.loadmat(ruta_comp)
    flexion_dedos_entrenamiento = interpolar_flexion_dedos(
        redimensionar_columna_datos_ecog(datos['train_dg'].astype('float64'))
    )
    espectrogramas_entrenamiento = normalizar_espectrogramas_a_db(
                   reducir_muestreo_espectrogramas(
                   calcular_espectrogramas(
                   filtrar_datos_ecog(
                   normalizar(
                   redimensionar_columna_datos_ecog(datos['train_data'].astype('float64'))))), nueva_fs=FS_SUBMUESTREO))
    # --- B. DATOS DE VALIDACIÓN/TEST ---
    print("\n>> Procesando Validación (Test)...")
    if os.path.exists(ruta_test):
        datos_test = scipy.io.loadmat(ruta_test)
        dg_test = datos_test['test_dg']
    else:
        # Algunos datasets no tienen el archivo de labels separado, ajusta si es necesario
        dg_test = datos['test_dg']

    flexion_dedos_validacion = interpolar_flexion_dedos(
        redimensionar_columna_datos_ecog(dg_test.astype('float64'))
    )
    espectrogramas_validacion = normalizar_espectrogramas_a_db(
                       reducir_muestreo_espectrogramas(
                       calcular_espectrogramas(
                       filtrar_datos_ecog(
                       normalizar(
                       redimensionar_columna_datos_ecog(datos['test_data'].astype('float64'))))), nueva_fs=FS_SUBMUESTREO))

    # --- C. RECORTAR POR RETRASO TEMPORAL ---
    print("\n>> Sincronizando (Time Delay)...")
    flexion_entrenamiento_recortada, espectro_entrenamiento_recortado = recortar_por_retraso_temporal(
        flexion_dedos_entrenamiento, espectrogramas_entrenamiento, retraso_temporal_segs, fs_actual
    )
    flexion_validacion_recortada, espectro_validacion_recortado = recortar_por_retraso_temporal(
        flexion_dedos_validacion, espectrogramas_validacion, retraso_temporal_segs, fs_actual
    )

    # --- D. ESCALADO (Machine Learning) ---
    print("\n>> Escalando datos para Machine Learning...")

    # 1. Escalar dedos (MinMaxScaler)
    escalador_dedos = MinMaxScaler()
    escalador_dedos.fit(flexion_entrenamiento_recortada.T)
    flexion_entrenamiento_escalada = escalador_dedos.transform(flexion_entrenamiento_recortada.T).T
    flexion_validacion_escalada = escalador_dedos.transform(flexion_validacion_recortada.T).T

    # 2. Escalar ECoG (RobustScaler)
    transformador_ecog = RobustScaler(unit_variance=True, quantile_range=(0.1, 0.9))
    transformador_ecog.fit(espectro_entrenamiento_recortado.T.reshape(-1, NUM_WAVELETS * num_canales))

    ecog_entrenamiento_escalado = transformador_ecog.transform(
        espectro_entrenamiento_recortado.T.reshape(-1, NUM_WAVELETS * num_canales)
    ).reshape(-1, NUM_WAVELETS, num_canales).T

    ecog_validacion_escalado = transformador_ecog.transform(
        espectro_validacion_recortado.T.reshape(-1, NUM_WAVELETS * num_canales)
    ).reshape(-1, NUM_WAVELETS, num_canales).T
    # --- E. GUARDADO FINAL ---
    print(f"Guardando datos finales del sujeto {sub_id} en Google Drive...")

    # Se añade el sufijo "_subX" a los archivos (ej. ecog_data_sub1.npy)
    sufijo_archivo = f"_sub{sub_id}"

    guardar_datos_procesados(ecog_entrenamiento_escalado, flexion_entrenamiento_escalada,
                             RUTA_GUARDADO, num_canales, val=False, agregar_nombre=sufijo_archivo)

    guardar_datos_procesados(ecog_validacion_escalado, flexion_validacion_escalada,
                             RUTA_GUARDADO, num_canales, val=True, agregar_nombre=sufijo_archivo)

    print(f"Sujeto {sub_id} procesado y guardado con éxito!")

print("\nTODOS LOS SUJETOS HAN SIDO PROCESADOS.")


SUJETO 1 (62 canales) 

>> Procesando Entrenamiento (Train)...
Interpolando flexión de dedos...
Calculando valores de frecuencia real...
Creando funciones de interpolación...
Interpolando con la frecuencia requerida...
Normalizando...
Normalizado...
Iniciando filtrado...
Setting up band-pass filter from 40 - 3e+02 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 40.00
- Lower transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 35.00 Hz)
- Upper passband edge: 300.00 Hz
- Upper transition bandwidth: 75.00 Hz (-6 dB cutoff frequency: 337.50 Hz)
- Filter length: 331 samples (0.331 s)

Frecuencias de ruido eliminadas...
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (

[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done  12 tasks      | elapsed:   36.7s
[Parallel(n_jobs=6)]: Done  62 out of  62 | elapsed:  2.1min finished
